In [0]:
import re
from datetime import datetime

base_raw_path = "s3://retail-sales-proj-t9/raw/"
base_archive_path = "s3://retail-sales-proj-t9/archive/"

folders = [
    "customers",
    "products",
    "stores",
    "sales"
]

for folder in folders:

    print(f"\nProcessing Folder: {folder}")

    raw_folder_path = f"{base_raw_path}{folder}/"

    files = dbutils.fs.ls(raw_folder_path)

    valid_files = []

    for file in files:

        file_name = file.name

        print("Checking:", file_name)

        match = re.match(
            r"(.+)_([0-9]{14})\.csv",
            file_name
        )

        if match:

            timestamp_str = match.group(2)

            timestamp = datetime.strptime(
                timestamp_str,
                "%d%m%Y%H%M%S"
            )

            valid_files.append(
                (timestamp, file)
            )

    if len(valid_files) > 1:

        sorted_files = sorted(
            valid_files,
            key=lambda x: x[0],
            reverse=True
        )

        latest_file = sorted_files[0][1]

        print("Keeping Latest:", latest_file.name)

        for _, old_file in sorted_files[1:]:

            source_path = old_file.path

            archive_path = (
                f"{base_archive_path}"
                f"{folder}/"
                f"{old_file.name}"
            )

            print("Archiving:", old_file.name)

            dbutils.fs.mv(
                source_path,
                archive_path
            )

    else:

        print("No archival needed")

print("\nIncremental archival completed successfully")